# 02 — Clean: Parse, Validate, and Spatial Join

Loads `data/raw/requests_{YEAR}.parquet`, applies cleaning and validation, performs a
point-in-polygon spatial join to assign each request a census tract GEOID, and saves
the result to `data/interim/requests_{YEAR}_clean.parquet`.

**Prerequisites:**
- `data/raw/requests_{YEAR}.parquet` (from `01_ingest.ipynb`)
- Census tract boundary file — see `geo_reference.md` for download instructions.
  Expected path: `data/raw/baltimore_tracts.geojson`
  Source: Census TIGER/Line or download via Census API.

**Output:** `data/interim/requests_{YEAR}_clean.parquet`

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path('.').resolve().parent / 'src'))

import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

from balt311.metrics import parse_timestamps, compute_days_to_close

YEAR = 2024
RAW_DIR    = Path('..') / 'data' / 'raw'
INTERIM    = Path('..') / 'data' / 'interim'
INTERIM.mkdir(exist_ok=True)

IN_FILE    = RAW_DIR / f'requests_{YEAR}.parquet'
TRACTS_GEO = RAW_DIR / 'baltimore_tracts.geojson'
OUT_FILE   = INTERIM / f'requests_{YEAR}_clean.parquet'

In [ ]:
df = pd.read_parquet(IN_FILE)
print(f'Loaded {len(df):,} rows from {IN_FILE.name}')
print(df.dtypes)

## 1. Parse timestamps and compute days_to_close

In [ ]:
df = parse_timestamps(df)
df = compute_days_to_close(df)

print('CreatedDate range:')
print(f'  {df["CreatedDate"].min()} → {df["CreatedDate"].max()}')
print(f'\ndays_to_close percentiles (closed records):')
closed = df.dropna(subset=['days_to_close'])
print(closed['days_to_close'].describe(percentiles=[.1,.25,.5,.75,.9,.95,.99]))

## 2. Coordinate coverage

In [ ]:
n = len(df)
valid_coords = df['Latitude'].notna() & df['Longitude'].notna() & (df['Latitude'] != 0)
print(f'Valid coordinates: {valid_coords.sum():,} / {n:,} ({100*valid_coords.mean():.1f}%)')
print(f'Missing / zero:    {(~valid_coords).sum():,}')

df_geo = df[valid_coords].copy()
df_nogeo = df[~valid_coords].copy()
print(f'\nProceeding with {len(df_geo):,} geocoded records.')

## 3. Spatial join to census tracts

Download Baltimore City census tract boundaries if not present:
```python
# One-time download via Census TIGER API (no key required)
import urllib.request
url = 'https://tigerweb.geo.census.gov/arcgis/rest/services/TIGERweb/tigerWMS_Current/MapServer/8/query?where=STATE%3D24+AND+COUNTY%3D510&outFields=GEOID,NAME&f=geojson'
urllib.request.urlretrieve(url, '../data/raw/baltimore_tracts.geojson')
```

In [ ]:
if not TRACTS_GEO.exists():
    raise FileNotFoundError(
        f'{TRACTS_GEO} not found. See the markdown cell above for the one-time download command.'
    )

tracts = gpd.read_file(TRACTS_GEO).to_crs('EPSG:4326')
print(f'Tracts loaded: {len(tracts)} polygons')
print(f'CRS: {tracts.crs}')
print(tracts[['GEOID','NAME']].head())

In [ ]:
gdf = gpd.GeoDataFrame(
    df_geo,
    geometry=gpd.points_from_xy(df_geo['Longitude'], df_geo['Latitude']),
    crs='EPSG:4326',
)

joined = gpd.sjoin(gdf, tracts[['GEOID', 'geometry']], how='left', predicate='within')
joined = joined.rename(columns={'GEOID': 'tract_geoid'})

no_tract = joined['tract_geoid'].isna().sum()
print(f'Joined: {len(joined):,} rows')
print(f'No tract match: {no_tract:,} ({100*no_tract/len(joined):.1f}%)')

## 4. Save

In [ ]:
# Drop geometry column before saving (not needed downstream; saves space)
out = pd.DataFrame(joined.drop(columns=['geometry', 'index_right'], errors='ignore'))
out.to_parquet(OUT_FILE, index=False)
print(f'Saved {len(out):,} rows → {OUT_FILE}')
print(f'Columns: {list(out.columns)}')